# Phase 12 — Moltbook-Basisrate

Zählt, wie oft Agenten-Posts in fremde Schrift kippen, und wie oft die
Begriffe auftauchen, die die Jacobi-Linse an Position 43 aufgemacht hat.
**Keine Ausbreitungsbehauptung** — nur die Basisrate, die man vorher kennen
muss.

Quelle sind **veröffentlichte** Sammlungen auf GitHub, kein eigener Crawler:
Moltbooks Nutzungsbedingungen untersagen automatisiertes Abrufen, und eine
Forschungsausnahme gibt es nicht. Die Plattform wird nicht angefasst.

Braucht **keine GPU**, läuft in jeder Colab-Laufzeit. ~3 min.

In [ ]:
# === KORPUS-BASISRATE: kippen Agenten-Posts in fremde Schrift? ==============
# Zaehlt, mehr nicht. KEINE Ausbreitungsbehauptung - die braucht Expositions-
# daten und ein Cox-Modell; hier geht es um die Basisrate, die man vorher
# kennen muss. Laeuft ohne GPU.
#
# Datenquelle: VEROEFFENTLICHTE Sammlungen auf GitHub, kein eigener Crawler.
# Moltbooks Nutzungsbedingungen untersagen automatisiertes Abrufen (von zwei
# unabhaengigen Dritten so protokolliert; das Wort "API" kommt in den Bedin-
# gungen nicht vor, es gibt also auch keine Forschungsausnahme). Wir ruehren
# die Plattform nicht an und nehmen, was andere publiziert haben.
import os, sys, json, re, time, glob, subprocess, collections
import numpy as np
QUELLEN = [("ExtraE113/moltbook_data",     "https://github.com/ExtraE113/moltbook_data"),
           ("kelkalot/moltbook-observatory","https://github.com/kelkalot/moltbook-observatory"),
           ("searchsim-org/moltbook-analysis","https://github.com/searchsim-org/moltbook-analysis")]
ZIEL = "/content/moltbook_dump"
MAXFILES = 400
LOKAL = globals().get("KORPUS_PFAD")     # zum Testen: auf ein Verzeichnis zeigen
# ---------------- Klassifikator (identisch zu Phase 11/12) ------------------
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),(0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will would can it on as at be by".split())
def _fremd(ch): return ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW)
def _srun(t,run=3):
    c=0
    for ch in t:
        if _fremd(ch):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def classify_answer(t):
    if not t or not t.strip(): return "empty"
    al=[ch for ch in t if ch.isalpha()]
    fo=[ch for ch in al if _fremd(ch)]
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
SW=("takeover","gloss","latin-switch(fr)")
# ---------------- Schluessel aus der Jacobi-Linse ---------------------------
# Nicht ausgedacht: das sind die Token, die die Linse an Position 43 aufgemacht
# hat (L27-L35), plus die Glossen-Klammer, die bei L15 auftauchte.
SCHLUESSEL = {
 "中文名": re.compile("中文名"),
 "native name": re.compile(r"\bnative name", re.I),
 "local name": re.compile(r"\blocal name", re.I),
 "localized/localised": re.compile(r"\blocali[sz]ed\b", re.I),
 "untranslated": re.compile(r"\buntranslated\b", re.I),
 "-language / /language": re.compile(r"[-/]language\b", re.I),
 "in <Sprache>": re.compile(r"\bin (Chinese|Japanese|Korean|Arabic|Russian|Hindi|Thai|Hebrew)\b", re.I),
}
RX_KLAMMER = re.compile(r"[A-Za-z][A-Za-z .'-]{0,40}\(([^)]{1,60})\)")
def gloss_klammer(text):
    """lateinisches Wort, direkt gefolgt von einer Klammer mit fremder Schrift -
       genau die Form, die unser Klassifikator als 'gloss' zaehlt"""
    if not text: return 0
    n=0
    for m in RX_KLAMMER.finditer(text):
        if any(_fremd(c) for c in m.group(1)): n+=1
    return n
def wilson(k,n,z=1.96):
    if n==0: return (0.0,0.0,0.0)
    import math
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n)
    h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
# ---------------- schema-tolerantes Einlesen --------------------------------
TEXTFELDER=("content","body","text","selftext","message","post_content")
TITELFELDER=("title","headline","subject")
AUTORFELDER=("author","agent","agent_name","username","creator","author_name","name")
ZEITFELDER=("created_at","created","timestamp","createdAt","time","date","posted_at")
def hole(d,felder):
    for f in felder:
        v=d.get(f)
        if isinstance(v,str) and v.strip(): return v
        if isinstance(v,dict):
            for g in ("name","username","id"):
                if isinstance(v.get(g),str) and v[g].strip(): return v[g]
    return None
def ist_post(d):
    return isinstance(d,dict) and (hole(d,TEXTFELDER) or hole(d,TITELFELDER))
def sammle(obj,out,tiefe=0):
    """laeuft durch beliebig verschachteltes JSON und sammelt alles, was wie
       ein Post oder Kommentar aussieht - das Schema der Dumps ist nicht
       dokumentiert, also wird nichts vorausgesetzt"""
    if tiefe>8: return
    if isinstance(obj,dict):
        if ist_post(obj): out.append(obj)
        for v in obj.values(): sammle(v,out,tiefe+1)
    elif isinstance(obj,list):
        for v in obj: sammle(v,out,tiefe+1)
def lade(pfad,maxfiles=MAXFILES):
    dateien=[]
    for ext in ("*.json","*.jsonl","*.ndjson"):
        dateien+=glob.glob(os.path.join(pfad,"**",ext),recursive=True)
    dateien=sorted(dateien)[:maxfiles]
    recs=[]; gelesen=0; fehler=0
    for f in dateien:
        try:
            with open(f,encoding="utf-8") as fh:
                roh=fh.read()
            if f.endswith((".jsonl",".ndjson")):
                for line in roh.splitlines():
                    line=line.strip()
                    if line:
                        try: sammle(json.loads(line),recs)
                        except Exception: pass
            else:
                sammle(json.loads(roh),recs)
            gelesen+=1
        except Exception:
            fehler+=1
    return recs,dateien,gelesen,fehler
# ---------------- Ausfuehrung -----------------------------------------------
AUS=globals().get("RUN_OUT") or ("/content/drive/MyDrive/WeirdChat_Runs/korpus_"
                                 +time.strftime("%Y%m%d-%H%M%S"))
try:
    if AUS.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
        from google.colab import drive; drive.mount("/content/drive")
except Exception: pass
os.makedirs(AUS,exist_ok=True)
LINES=[]
def P(s=""): LINES.append(str(s))
FEHLER=None
try:
    P("MOLTBOOK-BASISRATE - %s"%time.strftime("%Y-%m-%d %H:%M:%S"))
    P("="*72)
    quelle=None
    if LOKAL and os.path.isdir(LOKAL):
        quelle=("lokal",LOKAL); pfad=LOKAL
    else:
        for name,url in QUELLEN:
            d=ZIEL+"_"+name.split("/")[-1]
            if not os.path.isdir(d):
                r=subprocess.run(["git","clone","--depth","1",url,d],
                                 capture_output=True,text=True,timeout=900)
                if r.returncode!=0:
                    P("  Klon fehlgeschlagen: %s (%s)"%(name,r.stderr.strip()[-90:]))
                    continue
            recs,_,_,_=lade(d,40)
            if len(recs)>=50:
                quelle=(name,url); pfad=d; break
            P("  %s geklont, aber nur %d Datensaetze gefunden - naechste Quelle"%(name,len(recs)))
    assert quelle, "keine der Quellen lieferte brauchbare Daten"
    P("Quelle: %s"%(quelle[0] if quelle[0]=="lokal" else "%s (%s)"%quelle))
    recs,dateien,gelesen,fehler=lade(pfad)
    P("Dateien: %d gefunden, %d gelesen, %d fehlerhaft | %d Datensaetze"
      %(len(dateien),gelesen,fehler,len(recs)))
    # entdoppeln
    seen=set(); posts=[]
    for r in recs:
        t=(hole(r,TITELFELDER) or "")+"\n"+(hole(r,TEXTFELDER) or "")
        k=t.strip()[:400]
        if not k or k in seen: continue
        seen.add(k)
        posts.append(dict(text=t.strip(),autor=hole(r,AUTORFELDER),zeit=hole(r,ZEITFELDER)))
    P("Nach Entdopplung: %d Posts | mit Autor %d | mit Zeitstempel %d"
      %(len(posts),sum(1 for p in posts if p["autor"]),sum(1 for p in posts if p["zeit"])))
    assert len(posts)>=50, "zu wenige Posts (%d)"%len(posts)
    # ---- Klassifikation ----------------------------------------------------
    kl=collections.Counter(); kipp=[]
    for p in posts:
        c=classify_answer(p["text"]); p["klasse"]=c; kl[c]+=1
        if c in SW: kipp.append(p)
    n=len(posts); k=len(kipp)
    pr,lo,hi=wilson(k,n)
    P(""); P("SCHRIFTWECHSEL IN AGENTEN-POSTS")
    P("  Basisrate: %d von %d = %.2f%%  [%.2f, %.2f]"%(k,n,100*pr,100*lo,100*hi))
    for c,v in kl.most_common():
        P("    %-18s %6d  %5.2f%%"%(c,v,100*v/n))
    # ---- Schluessel aus der Linse -------------------------------------------
    P(""); P("SCHLUESSEL AUS DER JACOBI-LINSE (Position 43, L27-L35)")
    P("  %-24s %8s %8s   %s"%("Schluessel","Posts","Anteil","in Kipp-Posts"))
    SCHL={}
    for name,rx in SCHLUESSEL.items():
        tr=[p for p in posts if rx.search(p["text"])]
        tk=sum(1 for p in tr if p["klasse"] in SW)
        SCHL[name]=dict(n=len(tr),kipp=tk)
        P("  %-24s %8d %7.2f%%   %d/%d = %s"
          %(name,len(tr),100*len(tr)/n,tk,len(tr),
            ("%.1f%%"%(100*tk/len(tr))) if tr else "-"))
    gl=[p for p in posts if gloss_klammer(p["text"])>0]
    P("  %-24s %8d %7.2f%%"%("Glossen-Klammer",len(gl),100*len(gl)/n))
    # ---- Anreicherung: kippen Posts mit Schluessel oefter? ------------------
    P(""); P("ANREICHERUNG (deskriptiv, KEIN Ausbreitungsbefund)")
    mit=[p for p in posts if any(rx.search(p["text"]) for rx in SCHLUESSEL.values())]
    ohne=[p for p in posts if p not in mit] if len(mit)<2000 else []
    km=sum(1 for p in mit if p["klasse"] in SW)
    ko=k-km; no=n-len(mit)
    if mit and no:
        pm,_,_=wilson(km,len(mit)); po,_,_=wilson(ko,no)
        P("  mit Schluessel : %d/%d = %.2f%%"%(km,len(mit),100*pm))
        P("  ohne Schluessel: %d/%d = %.2f%%"%(ko,no,100*po))
        P("  Verhaeltnis: %.1fx"%(pm/max(po,1e-9)))
        P("  (Das ist Ko-Auftreten im selben Post, nicht Uebertragung zwischen")
        P("   Agenten. Fuer Uebertragung braucht es Exposition ueber die Zeit.)")
    # ---- Verteilung ueber Agenten ------------------------------------------
    if any(p["autor"] for p in posts):
        pro=collections.Counter(p["autor"] for p in kipp if p["autor"])
        alle=collections.Counter(p["autor"] for p in posts if p["autor"])
        P(""); P("VERTEILUNG UEBER AGENTEN")
        P("  Agenten gesamt: %d | Agenten mit mindestens einem Kipp: %d"
          %(len(alle),len(pro)))
        P("  Top-Kipper:")
        for a,c in pro.most_common(8):
            P("    %-28s %3d von %3d Posts"%(str(a)[:28],c,alle[a]))
        if len(pro)>=2:
            konz=sum(c for _,c in pro.most_common(3))/max(k,1)
            P("  Anteil der drei staerksten Agenten am Gesamtkipp: %.1f%%"%(100*konz))
            P("  (Hoch = ein paar Agenten tragen alles; das waere ein Hinweis auf")
            P("   Eigenart einzelner Agenten und nicht auf ein Populationsphaenomen.)")
    # ---- Beispiele ----------------------------------------------------------
    P(""); P("BEISPIELE (bis zu 8 Kipp-Posts, gekuerzt)")
    for p in kipp[:8]:
        t=" ".join(p["text"].split())
        P("  [%s | %s] %s"%(p["klasse"],str(p["autor"])[:18],t[:150]))
    P(""); P("(Deskriptiv. Basisrate und Ko-Auftreten, keine Ausbreitung. Quelle ist")
    P(" eine veroeffentlichte Sammlung, die Plattform wurde nicht abgefragt.)")
    KORPUS_RESULTS=dict(quelle=quelle[0],n_posts=n,n_kipp=k,rate=pr,ci=[lo,hi],
                        klassen=dict(kl),schluessel=SCHL,
                        n_glossklammer=len(gl),
                        n_agenten=len({p["autor"] for p in posts if p["autor"]}))
    globals()["KORPUS_RESULTS"]=KORPUS_RESULTS
    with open(os.path.join(AUS,"KORPUS_RESULTS.json"),"w",encoding="utf-8") as f:
        json.dump(KORPUS_RESULTS,f,ensure_ascii=False,indent=1)
    with open(os.path.join(AUS,"kipp_posts.jsonl"),"w",encoding="utf-8") as f:
        for p in kipp: f.write(json.dumps(p,ensure_ascii=False)+"\n")
except Exception as e:
    import traceback; FEHLER=traceback.format_exc(); P(""); P("ABBRUCH: %s"%e); P(FEHLER)
finally:
    with open(os.path.join(AUS,"bericht_korpus.txt"),"w",encoding="utf-8") as f:
        f.write("\n".join(LINES))
    print("GESCHRIEBEN NACH:",AUS)
    print("\n".join(LINES[-45:]) if not FEHLER else FEHLER.splitlines()[-1])
